# Notebook 10 — Backtesting de Portafolio Diversificado → MongoDB
### Investing AI · iDeSo · UNMSM

**Objetivo:** Simular, día por día y sobre datos históricos reales, cómo le habría ido a un
portafolio diversificado entre los 5 tickers mineros del proyecto, aplicando las señales
BUY/SELL de un clasificador (`historico_senales` en `predicciones`) dentro de cada "sleeve"
de capital, con los pesos de asignación calculados por el Notebook 9 (Markowitz).

**Este notebook alimenta DOS pestañas del frontend a la vez:**
- **Backtesting**: reporte técnico completo (Sharpe, Sortino, Max Drawdown, Win Rate,
  Profit Factor, lista de trades, curva de equity).
- **Portafolio**: vista resumida del mismo resultado (P&L actual, posiciones por ticker,
  distribución real del capital, curva de equity).

**Interactividad:** se precalculan las **15 combinaciones** de `modelo` (SVC, LSTM, BiLSTM,
GRU, SimpleRNN) × `perfil_riesgo` (conservador, moderado, agresivo), para que el frontend
pueda cambiar entre ellas sin volver a ejecutar nada — mismo patrón que el selector de
arquitecturas RNN.

**Costos de operación:** comisión 0.10% + slippage 0.05% por operación (compra y venta),
igual que ya anunciaba el encabezado de la pestaña Backtesting del frontend.

**Prerequisitos:**
- Notebook 1 (Ingesta) — `precios_ohlcv` poblado.
- Notebooks 2/3/4/5/6 (SVC, LSTM, BiLSTM, GRU, SimpleRNN) — `predicciones` con
  `historico_senales` para los 5 tickers, en cada uno de los 5 modelos.
- Notebook 9 (Markowitz) — `estrategias` con los pesos de los 3 perfiles de riesgo
  (se usa el horizonte `1y` como referencia de asignación).

**Salida:** Colección `backtests` en MongoDB (15 documentos), más `datos_backtests.json`.

## Paso 1 — Instalación de librerías

In [ ]:
!pip install "pymongo[srv]" --quiet
print("Librerías instaladas correctamente")

## Paso 2 — Conectar a MongoDB Atlas

In [ ]:
from google.colab import userdata
from pymongo import MongoClient

MONGO_URI = userdata.get("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["spbi"]

client.admin.command("ping")
print("✓ Conexión a MongoDB Atlas establecida")
print("  Base de datos:", db.name)

## Paso 3 — Configuración: tickers, modelos, perfiles y semilla

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
MODELOS = ["SVC", "LSTM", "BiLSTM", "GRU", "SimpleRNN"]
PERFILES = ["conservador", "moderado", "agresivo"]
HORIZONTE_PESOS = "1y"   # qué horizonte de Notebook 9 se usa como fuente de los pesos

CAPITAL_BASE = 100_000.0   # capital de referencia; el frontend escala proporcionalmente
COMISION_PCT  = 0.0010     # 0.10% por operación
SLIPPAGE_PCT  = 0.0005     # 0.05% por operación
TASA_LIBRE_RIESGO = 0.045  # anual, igual que en el Notebook 9 (para Sharpe/Sortino)

print(f"✓ SEED={SEED}")
print(f"  Tickers : {TICKERS}")
print(f"  Modelos : {MODELOS}")
print(f"  Perfiles: {PERFILES}")
print(f"  Capital base: ${CAPITAL_BASE:,.0f} | Comisión: {COMISION_PCT:.2%} | Slippage: {SLIPPAGE_PCT:.2%}")

## Paso 4 — Cargar pesos de asignación (Notebook 9) por perfil de riesgo

Se usa el horizonte `1y` de la colección `estrategias` como fuente de los pesos —
es el horizonte más representativo del histórico disponible en el proyecto (~1 año).

In [ ]:
def cargar_pesos_perfil(perfil):
    """
    Lee el documento de 'estrategias' (Notebook 9) para un perfil de riesgo,
    con horizonte='1y', y devuelve un dict {ticker: peso_decimal}.
    Si no existe (Notebook 9 no ejecutado), usa pesos iguales como respaldo.
    """
    doc = db["estrategias"].find_one({"perfil_riesgo": perfil, "horizonte": HORIZONTE_PESOS})
    if not doc:
        print(f"  ⚠ No se encontró 'estrategias' para perfil={perfil}, horizonte={HORIZONTE_PESOS}. "
              f"Usando pesos iguales como respaldo.")
        return {t: 1.0 / len(TICKERS) for t in TICKERS}

    return {a["ticker"]: a["asignacion_pct"] / 100.0 for a in doc["activos"]}

PESOS_POR_PERFIL = {perfil: cargar_pesos_perfil(perfil) for perfil in PERFILES}

print("✓ Pesos de asignación cargados desde 'estrategias' (Notebook 9)")
for perfil, pesos in PESOS_POR_PERFIL.items():
    resumen = " | ".join(f"{t}={p:.1%}" for t, p in pesos.items())
    print(f"  [{perfil:11s}] {resumen}")

## Paso 5 — Cargar señales históricas (`historico_senales`) por ticker y modelo

Se reutilizan las predicciones ya calculadas y guardadas por los Notebooks 2/3/4/5/6 —
este notebook NO reentrena ningún modelo, solo simula operaciones siguiendo las señales
que ya existen en MongoDB.

In [ ]:
def cargar_senales(ticker, modelo):
    """
    Lee 'predicciones' para un ticker+modelo y devuelve un DataFrame con
    columnas [fecha, precio, senal] ordenado cronológicamente.
    Devuelve DataFrame vacío si no existe la combinación.
    """
    doc = db["predicciones"].find_one({"ticker": ticker, "modelo": modelo})
    if not doc or not doc.get("historico_senales"):
        return pd.DataFrame(columns=["fecha", "precio", "senal"])

    df = pd.DataFrame(doc["historico_senales"])
    df = df.rename(columns={"prediccion": "senal"})[["fecha", "precio", "senal"]]
    df["fecha"] = pd.to_datetime(df["fecha"])
    df = df.sort_values("fecha").drop_duplicates(subset="fecha").reset_index(drop=True)
    return df

# Verificación rápida
_test = cargar_senales("BVN", "LSTM")
print(f"✓ Función cargar_senales definida")
print(f"  Prueba BVN + LSTM: {len(_test)} señales históricas")
if len(_test) > 0:
    print(_test.tail(3).to_string(index=False))

## Paso 6 — Motor de backtesting: simular un ticker (un "sleeve" de capital)

Recorre las señales día por día. Cambia de posición solo en las TRANSICIONES
(pasar de no-tener a BUY, o de tener a SELL) — no opera de nuevo si la señal se
repite mientras ya está en la posición correspondiente, evitando ruido de trades
redundantes. Aplica comisión y slippage en cada operación real.

In [ ]:
def backtest_ticker(df_senales, capital_sleeve):
    """
    Simula un solo ticker con un capital dedicado (capital_sleeve).
    Devuelve:
      - curva_valor: lista de {fecha, valor} día a día (mark-to-market)
      - trades: lista de operaciones cerradas {fecha_entrada, fecha_salida,
                precio_entrada, precio_salida, retorno_pct, duracion_dias}
      - posicion_final: dict con el estado al cierre del backtest
    """
    if df_senales.empty:
        return [], [], {"cantidad": 0, "precio_entrada": None, "en_posicion": False}

    efectivo = capital_sleeve
    acciones = 0.0
    precio_entrada = None
    fecha_entrada = None
    trades = []
    curva_valor = []

    for _, fila in df_senales.iterrows():
        fecha, precio, senal = fila["fecha"], float(fila["precio"]), fila["senal"]

        # ── Transición: comprar ──────────────────────────────────────
        if senal == "BUY" and acciones == 0:
            precio_ejec = precio * (1 + SLIPPAGE_PCT)          # slippage en contra al comprar
            costo_bruto = efectivo
            comision    = costo_bruto * COMISION_PCT
            monto_neto  = costo_bruto - comision
            acciones    = monto_neto / precio_ejec
            efectivo    = 0.0
            precio_entrada = precio_ejec
            fecha_entrada  = fecha

        # ── Transición: vender ───────────────────────────────────────
        elif senal == "SELL" and acciones > 0:
            precio_ejec = precio * (1 - SLIPPAGE_PCT)          # slippage en contra al vender
            bruto       = acciones * precio_ejec
            comision    = bruto * COMISION_PCT
            efectivo    = bruto - comision

            retorno_pct = (precio_ejec - precio_entrada) / precio_entrada * 100
            duracion    = (fecha - fecha_entrada).days
            trades.append({
                "fecha_entrada": fecha_entrada, "fecha_salida": fecha,
                "precio_entrada": round(precio_entrada, 4), "precio_salida": round(precio_ejec, 4),
                "retorno_pct": round(retorno_pct, 2), "duracion_dias": duracion
            })
            acciones = 0.0
            precio_entrada = None
            fecha_entrada = None

        # ── Marcar a mercado el valor del día (en posición o en efectivo) ──
        valor_dia = efectivo + acciones * precio
        curva_valor.append({"fecha": fecha, "valor": valor_dia})

    ultimo_precio = float(df_senales.iloc[-1]["precio"])
    posicion_final = {
        "cantidad"       : round(acciones, 6),
        "precio_entrada" : round(precio_entrada, 4) if precio_entrada else None,
        "precio_actual"  : ultimo_precio,
        "en_posicion"    : acciones > 0
    }

    return curva_valor, trades, posicion_final

print("✓ Función backtest_ticker definida")

## Paso 7 — Motor de backtesting: combinar los 5 tickers en un portafolio

Cada ticker recibe su "sleeve" de capital según el peso del perfil de riesgo
(Notebook 9). Las curvas de valor individuales se combinan por fecha (unión de
fechas, rellenando con el último valor conocido) para obtener la curva de
equity total del portafolio.

In [ ]:
def combinar_curvas(curvas_por_ticker):
    """
    Combina las curvas de valor de los 5 tickers (cada una con sus propias
    fechas) en una sola curva de equity total, alineando por fecha y
    rellenando huecos con el último valor conocido (forward-fill).
    """
    series = {}
    for ticker, curva in curvas_por_ticker.items():
        if not curva:
            continue
        df = pd.DataFrame(curva).set_index("fecha")["valor"]
        series[ticker] = df

    if not series:
        return []

    df_total = pd.DataFrame(series).sort_index().ffill().bfill()
    df_total["total"] = df_total.sum(axis=1)

    return [
        {"fecha": fecha.strftime("%Y-%m-%d"), "valor": round(float(valor), 2)}
        for fecha, valor in df_total["total"].items()
    ]

print("✓ Función combinar_curvas definida")

## Paso 8 — Cálculo de métricas de backtesting (Sharpe, Sortino, Drawdown, Win Rate...)

In [ ]:
def calcular_metricas(curva_equity, capital_inicial, todos_los_trades):
    """
    Calcula las métricas estándar de backtesting sobre la curva de equity
    combinada y la lista completa de trades cerrados.
    """
    if len(curva_equity) < 2:
        return {
            "total_return_pct": 0.0, "retorno_anualizado_pct": 0.0,
            "sharpe_ratio": 0.0, "sortino_ratio": 0.0, "max_drawdown_pct": 0.0,
            "win_rate_pct": 0.0, "profit_factor": 0.0, "total_trades": 0
        }

    valores = np.array([p["valor"] for p in curva_equity])
    valor_final = valores[-1]

    # ── Retorno total y anualizado ────────────────────────────────
    total_return = (valor_final - capital_inicial) / capital_inicial
    n_dias = len(valores)
    años = max(n_dias / 252, 1e-6)
    retorno_anualizado = (1 + total_return) ** (1 / años) - 1

    # ── Retornos diarios (para Sharpe/Sortino) ────────────────────
    retornos_diarios = np.diff(valores) / valores[:-1]
    media_diaria = retornos_diarios.mean()
    std_diaria   = retornos_diarios.std()

    sharpe = 0.0
    if std_diaria > 0:
        sharpe = ((media_diaria * 252) - TASA_LIBRE_RIESGO) / (std_diaria * np.sqrt(252))

    retornos_negativos = retornos_diarios[retornos_diarios < 0]
    std_downside = retornos_negativos.std() if len(retornos_negativos) > 0 else 0.0
    sortino = 0.0
    if std_downside > 0:
        sortino = ((media_diaria * 252) - TASA_LIBRE_RIESGO) / (std_downside * np.sqrt(252))

    # ── Máximo drawdown ─────────────────────────────────────────────
    pico = np.maximum.accumulate(valores)
    drawdown = (valores - pico) / pico
    max_drawdown = drawdown.min()

    # ── Win rate / profit factor (sobre trades cerrados) ────────────
    n_trades = len(todos_los_trades)
    if n_trades > 0:
        ganadores = [t["retorno_pct"] for t in todos_los_trades if t["retorno_pct"] > 0]
        perdedores = [t["retorno_pct"] for t in todos_los_trades if t["retorno_pct"] <= 0]
        win_rate = len(ganadores) / n_trades * 100
        suma_ganancias = sum(ganadores) if ganadores else 0.0
        suma_perdidas  = abs(sum(perdedores)) if perdedores else 0.0
        profit_factor = (suma_ganancias / suma_perdidas) if suma_perdidas > 0 else (suma_ganancias if suma_ganancias > 0 else 0.0)
    else:
        win_rate = 0.0
        profit_factor = 0.0

    return {
        "total_return_pct"       : round(total_return * 100, 2),
        "retorno_anualizado_pct" : round(retorno_anualizado * 100, 2),
        "sharpe_ratio"           : round(float(sharpe), 2),
        "sortino_ratio"          : round(float(sortino), 2),
        "max_drawdown_pct"       : round(float(max_drawdown) * 100, 2),
        "win_rate_pct"           : round(win_rate, 1),
        "profit_factor"          : round(profit_factor, 2),
        "total_trades"           : n_trades
    }

print("✓ Función calcular_metricas definida")

## Paso 9 — Pipeline completo: backtest de portafolio para una combinación (modelo, perfil)

In [ ]:
def backtest_portafolio(modelo, perfil):
    """
    Ejecuta el backtest diversificado completo para una combinación
    modelo (señales) x perfil (pesos de asignación de Notebook 9).
    """
    pesos = PESOS_POR_PERFIL[perfil]
    curvas_por_ticker = {}
    todos_los_trades = []
    posiciones_finales = []

    for ticker in TICKERS:
        capital_sleeve = CAPITAL_BASE * pesos.get(ticker, 0.0)
        df_senales = cargar_senales(ticker, modelo)

        curva, trades, posicion = backtest_ticker(df_senales, capital_sleeve)
        curvas_por_ticker[ticker] = curva

        for t in trades:
            t["ticker"] = ticker
        todos_los_trades.extend(trades)

        valor_actual = (posicion["cantidad"] * posicion["precio_actual"]) if posicion["en_posicion"] else capital_sleeve
        pnl_usd = valor_actual - capital_sleeve if capital_sleeve > 0 else 0.0
        pnl_pct = (pnl_usd / capital_sleeve * 100) if capital_sleeve > 0 else 0.0

        posiciones_finales.append({
            "ticker"        : ticker,
            "peso_asignado_pct": round(pesos.get(ticker, 0.0) * 100, 2),
            "capital_inicial_sleeve": round(capital_sleeve, 2),
            "cantidad"      : posicion["cantidad"],
            "precio_entrada": posicion["precio_entrada"],
            "precio_actual" : posicion["precio_actual"],
            "valor_actual"  : round(valor_actual, 2),
            "pnl_usd"       : round(pnl_usd, 2),
            "pnl_pct"       : round(pnl_pct, 2),
            "en_posicion"   : posicion["en_posicion"]
        })

    curva_total = combinar_curvas(curvas_por_ticker)

    # Ordenar trades cronológicamente y numerarlos para la tabla del frontend
    todos_los_trades.sort(key=lambda t: t["fecha_salida"])
    for i, t in enumerate(todos_los_trades, start=1):
        t["numero"] = i
        t["fecha_entrada"] = t["fecha_entrada"].strftime("%Y-%m-%d")
        t["fecha_salida"]  = t["fecha_salida"].strftime("%Y-%m-%d")

    metricas = calcular_metricas(curva_total, CAPITAL_BASE, todos_los_trades)

    return curva_total, todos_los_trades, posiciones_finales, metricas

print("✓ Función backtest_portafolio definida")

## Paso 10 — Ejecutar las 15 combinaciones (5 modelos × 3 perfiles) y guardar en MongoDB

In [ ]:
print("=" * 65)
print("  BACKTESTING DE PORTAFOLIO — 15 COMBINACIONES (MODELO × PERFIL)")
print("=" * 65)

resultados = []

for modelo in MODELOS:
    for perfil in PERFILES:
        curva_total, trades, posiciones, metricas = backtest_portafolio(modelo, perfil)

        documento = {
            "modelo"            : modelo,
            "perfil_riesgo"     : perfil,
            "horizonte_pesos"   : HORIZONTE_PESOS,
            "capital_base"      : CAPITAL_BASE,
            "comision_pct"      : COMISION_PCT,
            "slippage_pct"      : SLIPPAGE_PCT,
            "fecha_inicio"      : curva_total[0]["fecha"] if curva_total else None,
            "fecha_fin"         : curva_total[-1]["fecha"] if curva_total else None,
            "metricas"          : metricas,
            "equity_curve"      : curva_total,
            "trades"            : trades,
            "posiciones_finales": posiciones,
            "created_at"        : datetime.now()
        }

        db["backtests"].delete_many({"modelo": modelo, "perfil_riesgo": perfil})
        db["backtests"].insert_one(documento)
        resultados.append(documento)

        m = metricas
        print(f"  [{modelo:<10} · {perfil:<11}] retorno={m['total_return_pct']:+.1f}% | "
              f"sharpe={m['sharpe_ratio']:.2f} | maxDD={m['max_drawdown_pct']:.1f}% | "
              f"winrate={m['win_rate_pct']:.0f}% | trades={m['total_trades']}")

print()
print("=" * 65)
print(f"  ✅ Pipeline completo: {len(resultados)}/15 combinaciones guardadas en 'backtests'")
print("=" * 65)

## Paso 11 — Verificación en MongoDB

In [ ]:
total_docs = db["backtests"].count_documents({})
print(f"Total de documentos en 'backtests': {total_docs} (esperado: 15)")
print()
print("Combinaciones almacenadas:")
for doc in db["backtests"].find({}, {"_id": 0, "modelo": 1, "perfil_riesgo": 1, "metricas": 1}):
    m = doc["metricas"]
    print(f"  {doc['modelo']:<10} · {doc['perfil_riesgo']:<11} → "
          f"retorno={m['total_return_pct']:+.1f}% | sharpe={m['sharpe_ratio']:.2f} | trades={m['total_trades']}")

## Paso 12 — Visualización rápida: curvas de equity por modelo (perfil moderado)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))
for modelo in MODELOS:
    doc = db["backtests"].find_one({"modelo": modelo, "perfil_riesgo": "moderado"})
    if doc and doc["equity_curve"]:
        fechas  = [p["fecha"] for p in doc["equity_curve"]]
        valores = [p["valor"] for p in doc["equity_curve"]]
        plt.plot(fechas, valores, label=modelo, linewidth=1.5)

plt.axhline(y=CAPITAL_BASE, color="gray", linestyle="--", linewidth=1, label="Capital inicial")
plt.title("Curvas de Equity — Perfil Moderado, por Modelo")
plt.xlabel("Fecha")
plt.ylabel("Valor del Portafolio (USD)")
plt.xticks(rotation=45)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("✓ Gráfico comparativo generado (perfil moderado, 5 modelos)")

## Paso 13 — Exportación a `datos_backtests.json`

In [ ]:
import json

def serializar(doc):
    limpio = {k: v for k, v in doc.items() if k not in ("_id", "created_at")}
    return limpio

export_data = {
    "modelos"       : MODELOS,
    "perfiles"      : PERFILES,
    "capital_base"  : CAPITAL_BASE,
    "comision_pct"  : COMISION_PCT,
    "slippage_pct"  : SLIPPAGE_PCT,
    "seed"          : SEED,
    "fecha_export"  : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "combinaciones" : [serializar(doc) for doc in db["backtests"].find({}, {"_id": 0})]
}

with open("datos_backtests.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f"✓ Archivo datos_backtests.json exportado ({len(export_data['combinaciones'])} combinaciones)")
print(f"  Tamaño aproximado: {len(json.dumps(export_data)) / 1024:.1f} KB")

try:
    from google.colab import files
    files.download("datos_backtests.json")
    print("✓ Descarga iniciada en el navegador")
except Exception as e:
    print(f"  (Descarga automática no disponible en este entorno: {e})")

## Resultado

La colección `backtests` contiene **15 simulaciones reales** (5 modelos × 3 perfiles de
riesgo), cada una con:

- **Curva de equity diaria** del portafolio combinado (5 tickers, ponderados según
  Markowitz del Notebook 9).
- **Lista completa de trades** (fecha entrada/salida, precio, retorno %, duración),
  con comisión 0.10% y slippage 0.05% aplicados en cada operación.
- **Métricas de backtesting**: retorno total, retorno anualizado, Sharpe, Sortino,
  Max Drawdown, Win Rate, Profit Factor, total de trades.
- **Posiciones finales por ticker**: cantidad, precio de entrada/actual, P&L individual —
  listo para la tabla "Posiciones" y el pie chart de distribución real del frontend.

**Este único notebook alimenta dos pestañas:**
- **Backtesting**: reporte técnico completo, con selector de modelo (y perfil).
- **Portafolio**: vista resumida (P&L, posiciones, distribución, curva de equity),
  reutilizando exactamente los mismos documentos de `backtests`.

**Pipeline completo:** `predicciones` (señales) + `estrategias` (pesos Markowitz) →
simulación día a día con costos reales → `backtests` (MongoDB) → `datos_backtests.json` ✓

**Siguiente paso:** agregar el endpoint `/api/backtests?modelo=&perfil_riesgo=` a la API,
y conectar `renderM7()` (Portafolio) y `renderM11()` (Backtesting) a datos reales.